# Solucion Docente: Regresion con Taxi Price Prediction

Flujo completo: carga, validacion, EDA, preparacion, entrenamiento, benchmarking y validacion final.

## Resumen de particiones entregadas

- Filas originales: 1000
- Filas con target no nulo usadas para modelado: 951
- Excluidas por Trip_Price nulo: 49
- Desarrollo 90%: 855
- Validacion 10%: 96

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

RANDOM_STATE = 42
TARGET_COL = "Trip_Price"
pd.set_option("display.max_columns", None)

ModuleNotFoundError: No module named 'numpy'

## Carga de datos en Colab

Usa URL RAW (si la tienes) o carga manual a /content.

In [ ]:
DEV_URL = ""
VAL_URL = ""
DEV_LOCAL_PATH = "/content/taxi_dev_90.csv"
VAL_LOCAL_PATH = "/content/taxi_validation_10.csv"

def load_csv_smart(remote_url: str, local_path: str) -> pd.DataFrame:
    if isinstance(remote_url, str) and remote_url.strip():
        print(f"Leyendo URL: {remote_url}")
        return pd.read_csv(remote_url)
    if os.path.exists(local_path):
        print(f"Leyendo local: {local_path}")
        return pd.read_csv(local_path)
    raise FileNotFoundError(f"No se encontro origen de datos: {local_path}")

dev_df = load_csv_smart(DEV_URL, DEV_LOCAL_PATH)
val_df = load_csv_smart(VAL_URL, VAL_LOCAL_PATH)

print("dev_df shape:", dev_df.shape)
print("val_df shape:", val_df.shape)

In [ ]:
def quick_checks(df: pd.DataFrame, name: str, target_col: str) -> None:
    print(f"\n=== {name} ===")
    print("Shape:", df.shape)
    print("Duplicados:", df.duplicated().sum())
    print("Nulos top 10:")
    print(df.isna().sum().sort_values(ascending=False).head(10))
    print("Dtypes:")
    print(df.dtypes)
    if target_col in df.columns:
        print(f"Nulos en {target_col}:", df[target_col].isna().sum())
    display(df.head())

quick_checks(dev_df, "Desarrollo 90%", TARGET_COL)
quick_checks(val_df, "Validacion 10%", TARGET_COL)

## EDA y calidad de datos

In [ ]:
def missing_profile(df: pd.DataFrame) -> pd.DataFrame:
    return df.isna().mean().sort_values(ascending=False).rename("missing_ratio").to_frame()

def iqr_outlier_summary(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        s = df[col].dropna()
        if s.empty:
            continue
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        low = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        outliers = ((s < low) | (s > high)).sum()
        rows.append({"feature": col, "outliers": int(outliers), "ratio": outliers / len(s)})
    return pd.DataFrame(rows).sort_values("ratio", ascending=False)

display(dev_df.describe(include="all"))
display(missing_profile(dev_df).head(15))

plt.figure(figsize=(8, 4))
sns.histplot(dev_df[TARGET_COL].dropna(), kde=True, bins=30)
plt.title("Distribucion de Trip_Price")
plt.show()

numeric_cols = dev_df.select_dtypes(include=["number"]).columns.tolist()
numeric_cols_no_target = [c for c in numeric_cols if c != TARGET_COL]
display(iqr_outlier_summary(dev_df, numeric_cols_no_target).head(15))

## Preparacion y split interno 80/20

In [ ]:
X = dev_df.drop(columns=[TARGET_COL]).copy()
y = dev_df[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

print("X_train", X_train.shape, "X_test", X_test.shape)

## Entrenamiento y benchmarking

In [ ]:
def regression_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict:
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

def train_and_benchmark(X_train, X_test, y_train, y_test, preprocessor):
    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=RANDOM_STATE),
        "Lasso": Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=10000),
        "RandomForest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    }

    rows = []
    fitted = {}

    for name, model in models.items():
        pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        met = regression_metrics(y_test, pred)
        rows.append({"model": name, **met})
        fitted[name] = pipe

    bench = pd.DataFrame(rows).sort_values(["RMSE", "MAE"]).reset_index(drop=True)
    return bench, fitted

benchmark_df, fitted_models = train_and_benchmark(X_train, X_test, y_train, y_test, preprocessor)
display(benchmark_df)

best_model_name = benchmark_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]
print("Mejor modelo:", best_model_name)

## Validacion final 10% (holdout)

Funcion automatizada que recibe el modelo ajustado y evalua su desempeno final.

In [ ]:
def evaluate_on_holdout_validation(fitted_model: Pipeline, validation_df: pd.DataFrame, target_col: str = TARGET_COL):
    if target_col not in validation_df.columns:
        raise ValueError(f"No existe la columna target: {target_col}")

    X_val = validation_df.drop(columns=[target_col]).copy()
    y_val = validation_df[target_col].copy()

    y_pred = fitted_model.predict(X_val)
    metrics = regression_metrics(y_val, y_pred)

    preds = validation_df.copy()
    preds["y_pred"] = y_pred
    preds["abs_error"] = (preds[target_col] - preds["y_pred"]).abs()

    return preds, metrics

val_predictions_df, val_metrics = evaluate_on_holdout_validation(best_model, val_df, TARGET_COL)
print("Metricas en validacion final:")
print(json.dumps(val_metrics, indent=2))
display(val_predictions_df.head())

## Buenas practicas usadas

- Separacion de etapas por secciones claras.
- Funciones reutilizables para checks, metricas y evaluacion final.
- Pipelines para evitar data leakage.
- Benchmarking con criterios comparables.